In [ ]:
import torch

In [ ]:
# load the names dataset from file
from pathlib import Path

data_dir = Path.cwd() / ".." / ".." / "data"


def load_names(path: Path) -> list[str]:
    with path.open("r") as f:
        return f.read().splitlines()


words = load_names(data_dir / "names.txt")
words[:4]

In [ ]:
from makemore.vocab import Vocab
import string

# build the vocabulary
vocab = Vocab.from_words([string.ascii_lowercase])
len(vocab)

In [ ]:
# build the dataset
import random
from makemore.data import make_dataset

BLOCK_SIZE = 3


def split(
    words: list[str],
    vocab: Vocab,
    *,
    block_size: int,
    splits: tuple[float, float, float],
    seed: int
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
]:
    """Split the dataset"""
    assert sum(v for v in splits) == 1.0, "broken precondition"

    random.seed(seed)
    random.shuffle(words)

    n1 = int(splits[0] * len(words))
    n2 = int((splits[0] + splits[1]) * len(words))

    X_train, Y_train = make_dataset(words[:n1], vocab, block_size)
    X_dev, Y_dev = make_dataset(words[n1:n2], vocab, block_size)
    X_test, Y_test = make_dataset(words[n2:], vocab, block_size)

    return X_train, Y_train, X_dev, Y_dev, X_test, Y_test


X_train, Y_train, X_dev, Y_dev, X_test, Y_test = split(
    words, vocab, block_size=BLOCK_SIZE, splits=(0.8, 0.1, 0.1), seed=42
)
print(X_train.shape, Y_train.shape)
print(X_dev.shape, Y_dev.shape)
print(X_test.shape, Y_test.shape)

In [ ]:
# the character embedding dimension
embedding_dimension = 10
# the number of neurons in the hidden layer
hidden_layer_size = 200

g = torch.Generator().manual_seed(2147483647)

# embedding layer
C = torch.randn(
    (len(vocab), embedding_dimension),
    generator=g,
)
# hidden layer
kaiming_init_scale = (5 / 3) / ((BLOCK_SIZE * embedding_dimension) ** 0.5)
W1 = (
    torch.randn(
        (
            BLOCK_SIZE * embedding_dimension,
            hidden_layer_size,
        ),
        generator=g,
    )
    * kaiming_init_scale
)
b1 = torch.randn(hidden_layer_size, generator=g) * 0.01
# output layer
W2 = (
    torch.randn(
        (hidden_layer_size, len(vocab)),
        generator=g,
    )
    * 0.01
)
b2 = (
    torch.randn(
        len(
            vocab,
        ),
        generator=g,
    )
    * 0.0
)

bngain = torch.ones((1, hidden_layer_size))
bnbias = torch.zeros((1, hidden_layer_size))

bnmean_running = torch.zeros((1, hidden_layer_size))
bnstd_running = torch.ones((1, hidden_layer_size))

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

print(f"kaiming init scale = {kaiming_init_scale}")

n_parameters = sum(p.nelement() for p in parameters)
print(f"total parameters = {n_parameters}")

for p in parameters:
    p.requires_grad = True

In [ ]:
import torch.nn.functional as F

max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
    # minibatch construct
    ix = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    Xb, Yb = X_train[ix], Y_train[ix]

    # --- forward pass

    # embed the characters into vectors
    emb = C[Xb]
    # concatenate the vectors
    embcat = emb.view(emb.shape[0], -1)

    # hidden layer pre-activation
    hpreact = embcat @ W1 + b1
    bnmeani = hpreact.mean(dim=0, keepdim=True)
    bnstdi = hpreact.std(dim=0, keepdim=True)
    hpreact = (hpreact - bnmeani) / bnstdi
    hpreact = bngain * hpreact + bnbias

    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

    # non-linearity (complete hidden layer)
    h = torch.tanh(hpreact)
    # output layer
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)  # loss function

    # --- backward pass

    for p in parameters:
        p.grad = None
    loss.backward()

    # --- update

    lr = 0.1 if i < 100000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad  # type: ignore

    # --- track stats

    if i % 10000 == 0:
        print(f"{i:7d}/{max_steps:7d}: {loss.item():.4f}")
    lossi.append(loss.log10().item())

In [ ]:
@torch.no_grad()
def split_loss(split) -> float:
    x, y = {
        "train": (X_train, Y_train),
        "val": (X_dev, Y_dev),
        "test": (X_test, Y_test),
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    # hpreact = (hpreact - hpreact.mean(dim=0, keepdim=True)) / hpreact.std(dim=0, keepdim=True)
    hpreact = (hpreact - bnmean_running) / bnstd_running
    hpreact = bngain * hpreact + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    return loss.item()


print(f"train = {split_loss('train')}")
print(f"val   = {split_loss('val')}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(lossi)